In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

"""
Script to calculate lift times separated by Test type (Test1 vs Test2).
It performs the following steps:
1. Searches for 'classes_log.csv' in subject folders.
2. Groups data into 'Test1' (test1a/b) and 'Test2' (test2a/b).
3. Calculates lift duration:
   - Identifies transitions from Lift (label != 0) to Rest (label == 0).
   - Computes Duration = (Next_TS - Curr_TS) + Start_Trans_Time - End_Trans_Time.
4. Saves results to 'lift_times_Test1.csv' and 'lift_times_Test2.csv'.
5. Prints statistics for each group.
"""

def parse_custom_timestamp(ts_str):
    """
    Converts timestamp format 'HH_MM_SS_mmm' to total milliseconds.
    """
    try:
        parts = ts_str.split('_')
        h, m, s, ms = map(int, parts)
        return (h * 3600 * 1000) + (m * 60 * 1000) + (s * 1000) + ms
    except Exception:
        return None

def calculate_lift_times_separated(base_repo_path, output_base_dir):
    # 1. Path Definitions
    base_path = Path(base_repo_path)
    output_base_path = Path(output_base_dir)
    
    # General output folder
    output_dir_lift = output_base_path / "all" / "lift_times"
    
    print(f"--- Starting Lift Times Analysis (Separated by Test) ---")
    print(f"Input: {base_path}")
    print(f"Output Folder: {output_dir_lift}\n")

    # Dictionary to separate data
    # "Test1" will collect test1a, test1b
    # "Test2" will collect test2a, test2b
    data_groups = {
        "Test1": [],
        "Test2": []
    }

    # 2. Find all Subject folders
    s_folders = sorted([f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')])
    
    if not s_folders:
        print("No subject folders found.")
        return

    # 3. Iterate over Subjects
    for subject_folder in s_folders:
        subject_name = subject_folder.name
        
        # Find sessions
        session_folders = [f for f in subject_folder.iterdir() if f.is_dir()]
        
        for session_folder in session_folders:
            session_name = session_folder.name.lower() # make lowercase for easy checks
            
            # Determine which group the session belongs to
            target_group = None
            if "test1" in session_name:
                target_group = "Test1"
            elif "test2" in session_name:
                target_group = "Test2"
            
            # If the session is neither test1 nor test2, skip it
            if not target_group:
                continue

            # Search for classes_log.csv
            csv_files = list(session_folder.rglob("classes_log.csv"))
            
            for csv_file in csv_files:
                try:
                    df = pd.read_csv(
                        csv_file, 
                        usecols=['timestamp', 'label', 'transition_time'],
                        dtype={'timestamp': str, 'label': int, 'transition_time': float},
                        skipinitialspace=True
                    )

                    if df.empty:
                        continue

                    # Timestamp conversion
                    df['ts_ms'] = df['timestamp'].apply(parse_custom_timestamp)
                    
                    # Start -> End Pair Logic
                    for i in range(len(df) - 1):
                        row_curr = df.iloc[i]
                        row_next = df.iloc[i+1]

                        # Detect transition from Lift (label != 0) to Rest (label == 0)
                        if row_curr['label'] != 0 and row_next['label'] == 0:
                            
                            ts_diff = row_next['ts_ms'] - row_curr['ts_ms']
                            
                            # transition_time is in seconds, convert to ms
                            trans_start_ms = row_curr['transition_time'] * 1000
                            trans_end_ms = row_next['transition_time'] * 1000
                            
                            duration_ms = ts_diff + trans_start_ms - trans_end_ms
                            
                            # Add to the correct group list
                            data_groups[target_group].append({
                                'Subject': subject_name,
                                'Session': session_folder.name, # Original name (e.g., test1a)
                                'Lift_Label': row_curr['label'],
                                'Duration_ms': duration_ms,
                                'Start_Time': row_curr['timestamp'],
                                'End_Time': row_next['timestamp']
                            })

                except Exception as e:
                    print(f"Error reading {csv_file}: {e}")

    # 4. Final Processing and Saving for each Group
    try:
        output_dir_lift.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print(f"Error creating output directory: {e}")
        return
    
    for group_name, data_list in data_groups.items():
        print(f"\n--- PROCESSING GROUP: {group_name} ---")
        
        if not data_list:
            print(f" > No data found for {group_name}.")
            continue
            
        df_results = pd.DataFrame(data_list)
        
        # Specific file name for the group
        file_name = f"lift_times_{group_name}.csv"
        full_path = output_dir_lift / file_name
        
        # Save CSV
        df_results.to_csv(full_path, index=False)
        print(f" > File saved: {full_path}")
        
        # Calculate Statistics
        durations = df_results['Duration_ms']
        mean_val = durations.mean()
        std_val = durations.std()
        
        print(f" > STATISTICS {group_name} (N={len(durations)} lifts)")
        print(f"   MEAN:       {mean_val:.2f} ms")
        print(f"   STD DEV:    {std_val:.2f} ms")

# --- EXECUTION ---
if __name__ == "__main__":
    
    # Specified Paths
    data_path = r"C:\Users\nicol\Thesis\DATA real time\_test_raw"
    output_path = r"C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data"
    
    calculate_lift_times_separated(data_path, output_path)

--- Inizio Analisi Tempi Lift (Separata per Test) ---
Input: C:\Users\nicol\Thesis\DATA real time\_test_raw
Output Folder: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\lift_times

Errore leggendo C:\Users\nicol\Thesis\DATA real time\_test_raw\S12\test2b\5\classes_log.csv: No columns to parse from file

--- ELABORAZIONE GRUPPO: Test1 ---
 > File salvato: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\lift_times\lift_times_Test1.csv
 > STATISTICHE Test1 (N=246 lifts)
   MEDIA:      3754.77 ms
   DEV. STD:   1670.46 ms

--- ELABORAZIONE GRUPPO: Test2 ---
 > File salvato: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\lift_times\lift_times_Test2.csv
 > STATISTICHE Test2 (N=146 lifts)
   MEDIA:      6236.39 ms
   DEV. STD:   2392.55 ms
